# Computing Bounds for OOD datasets

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(['science'])
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams.update({'font.size': 10})
colors = plt.colormaps['tab20']

In [2]:
bound = lambda k, mu, s0: upper(k,mu, s0) - lower(k,mu,s0)

def upper(k,mu, s0, m=None, tol=1e-20, maxfloat=1e6):
    c = k*(1+mu)
    assert np.array(c).squeeze().ndim == 0 or np.array(s0).squeeze().ndim == 0
    if np.array(c).squeeze().ndim > 0:
        assert c.ndim == 1
        return np.array([upper(a, s0, m, tol) for a in c])
    if np.array(s0).squeeze().ndim > 0:
        assert s0.ndim == 1
        return np.array([upper(c, a, m, tol) for a in s0])
    if s0 < 0:
        return np.nan
    r = 0
    t = 1
    i = -1
    overflowcount = 0
    while True:
        i += 1
        if i > 0:
            t *= s0/i
        a = t * (c + s0)/(c + i)
        if i > 0 and np.abs(a/r) < tol:
            break
        r += a
        if m is not None and i >= m:
            break
        if r > maxfloat:
            r /= maxfloat
            t /= maxfloat
            overflowcount += 1
    r = np.exp(np.log(r) + overflowcount*np.log(maxfloat) - s0)
    return r

def lower(k,mu, s0):
    return (k*(1+mu) + s0)/(k+1+(k-1)*mu + s0)

In [3]:
def calc_k_mu_s0(p,y_hat):
    if np.sum(y_hat) == 0:
        print("Null Output")
        return None, None, None
    k = np.sum(y_hat)
    mu = np.sum(y_hat*p)/k
    s0 = np.sum((1-y_hat)*p)
    return k, mu, s0

In [4]:
from collections import defaultdict
dict_lower_bounds = defaultdict(list)
dict_upper_bounds = defaultdict(list)
dict_bounds = defaultdict(list)
dict_delta_bounds = defaultdict(list)
dict_epsilon_bounds = defaultdict(list)

In [5]:
import pickle
### MSWML
tresh = 0.35

task ='mswml_ood' # Change to 'refuge' for the other dataset, and to 'mswml' for MSWML dataset. For OOD evaluation, use 'mswml_ood'
if 'mswml' not in task:
    data = np.load(f'./data/{task}.npz')
    y = data['y']
    p_hat = data['p_hat']
    #y_hat = data['y_hat'] # This line should be COMMENTED in case y_hat is not given
elif 'ood' in task:
    p_hat = pickle.load(open('./data/mswml/dev_out_predictions.pkl', 'rb'))
    y = pickle.load(open('./data/mswml/dev_out_gt.pkl', 'rb'))
else:
    p_hat = pickle.load(open('./data/mswml/eval_in_predictions.pkl', 'rb'))
    y = pickle.load(open('./data/mswml/eval_in_gt.pkl', 'rb'))
    
    
print(len(p_hat))

list_zeros = []
for i,p_h in enumerate(p_hat):
    binary_p_hat =  p_h>0.35
    if np.sum(binary_p_hat)==0:
        list_zeros.append(i)
        
for i in list_zeros:
    p_hat.pop(i)
    y.pop(i)
    
print(len(p_hat))



for p_hat_i in p_hat:
    p_hat_i = np.array(p_hat_i)
    y_hat_i = p_hat_i>tresh
    k, mu, s0 = calc_k_mu_s0(p_hat_i,y_hat_i)
    if (k != None) and (mu!= None) and (s0!= None):
        dict_lower_bounds['MSWML'].append(lower(k,mu, s0))
        dict_upper_bounds['MSWML'].append(upper(k,mu, s0))
        dict_delta_bounds['MSWML'].append(upper(k,mu, s0)-lower(k,mu, s0))
        dict_bounds['MSWML'].append(lower(k,mu, s0))
        dict_bounds['MSWML'].append(upper(k,mu, s0))
        dict_epsilon_bounds['MSWML'].append(max((1/lower(k,mu, s0)-1),(1-1/upper(k,mu, s0))))

25
25


In [6]:
### POLYP PVT
tresh = 0.5
task ='polyp_ood' # Change to 'refuge' for the other dataset, and to 'mswml' for MSWML dataset. For OOD evaluation, use 'mswml_ood'
if 'mswml' not in task:
    data = np.load(f'./data/{task}.npz')
    y = data['y']
    p_hat = data['p_hat']
    #y_hat = data['y_hat'] # This line should be COMMENTED in case y_hat is not given
elif 'ood' in task:
    p_hat = pickle.load(open('./data/mswml/dev_out_predictions.pkl', 'rb'))
    y = pickle.load(open('./data/mswml/dev_out_gt.pkl', 'rb'))
else:
    p_hat = pickle.load(open('./data/mswml/eval_in_predictions.pkl', 'rb'))
    y = pickle.load(open('./data/mswml/eval_in_gt.pkl', 'rb'))
    
zero_preds = np.all(y == 0, axis=(-2,-1)).flatten()
p_hat= p_hat[~zero_preds]
y = y[~zero_preds]

#binary_y_hat =  y_hat>0.5
#zero_preds = np.all(binary_y_hat == 0, axis=(-2,-1)).flatten()
#y_hat= y_hat[~zero_preds]
#y = y[~zero_preds]


print(len(y), len(p_hat))

for p_hat_i in p_hat:
    y_hat_i = p_hat_i>tresh
    k, mu, s0 = calc_k_mu_s0(p_hat_i,y_hat_i)
    if (k != None) and (mu!= None) and (s0!= None):
        dict_lower_bounds['Polyp'].append(lower(k,mu, s0))
        dict_upper_bounds['Polyp'].append(upper(k,mu, s0))
        dict_delta_bounds['Polyp'].append(upper(k,mu, s0)-lower(k,mu, s0))
        dict_bounds['Polyp'].append(lower(k,mu, s0))
        dict_bounds['Polyp'].append(upper(k,mu, s0))
        dict_epsilon_bounds['Polyp'].append(max((1/lower(k,mu, s0)-1),(1-1/upper(k,mu, s0))))

698 698
Null Output


In [7]:
### REFUGE
tresh = 0.5
task ='optic_cup_ood' # Change to 'refuge' for the other dataset, and to 'mswml' for MSWML dataset. For OOD evaluation, use 'mswml_ood'
if 'mswml' not in task:
    data = np.load(f'./data/{task}.npz')
    y = data['y']
    p_hat = data['p_hat']
    #y_hat = data['y_hat'] # This line should be COMMENTED in case y_hat is not given
elif 'ood' in task:
    p_hat = pickle.load(open('./data/mswml/dev_out_predictions.pkl', 'rb'))
    y = pickle.load(open('./data/mswml/dev_out_gt.pkl', 'rb'))
else:
    p_hat = pickle.load(open('./data/mswml/eval_in_predictions.pkl', 'rb'))
    y = pickle.load(open('./data/mswml/eval_in_gt.pkl', 'rb'))
    

y.shape, p_hat.shape


print(len(y), len(p_hat))

for p_hat_i in p_hat:
    y_hat_i = p_hat_i>tresh
    k, mu, s0 = calc_k_mu_s0(p_hat_i,y_hat_i)
    if (k != None) and (mu!= None) and (s0!= None):
        dict_lower_bounds['Optic Cup'].append(lower(k,mu, s0))
        dict_upper_bounds['Optic Cup'].append(upper(k,mu, s0))
        dict_delta_bounds['Optic Cup'].append(upper(k,mu, s0)-lower(k,mu, s0))
        dict_bounds['Optic Cup'].append(lower(k,mu, s0))
        dict_bounds['Optic Cup'].append(upper(k,mu, s0))
        dict_epsilon_bounds['Optic Cup'].append(max((1/lower(k,mu, s0)-1),(1-1/upper(k,mu, s0))))

1438 1438
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output
Null Output


In [8]:
import pandas as pd

def generate_latex_table(data):
    # Convert dictionary to DataFrame, handling unequal lengths by filling with NaN
    max_length = max(len(v) for v in data.values())
    padded_data = {key: values + [np.nan] * (max_length - len(values)) for key, values in data.items()}
    df = pd.DataFrame(padded_data)
    
    summary_stats = df.agg(['min', 'max', 'median']).transpose()
    
    # Format the values to 5 decimal places
    summary_stats = summary_stats.round(5)
    
    # Generate the LaTeX table
    latex_table = "\\begin{table}[h!]\n\\centering\n\\begin{tabular}{|l|r|r|r|}\n\\hline\n"
    latex_table += "Dataset & Min & Max & Median \\\\\n\\hline\n"
    for col in summary_stats.index:
        min_val = f"{summary_stats.loc[col, 'min']:.5f}"
        max_val = f"{summary_stats.loc[col, 'max']:.5f}"
        median_val = f"{summary_stats.loc[col, 'median']:.5f}"
        latex_table += f"{col} & {min_val} & {max_val} & {median_val} \\\\\n"
    latex_table += "\\hline\n\\end{tabular}\n\\caption{Summary Statistics Table}\n\\label{tab:summary_stats}\n\\end{table}"
    
    return latex_table

In [9]:
import numpy as np
import pandas as pd

def scientific_notation_around_one(value):
    """Format a number close to 1 in scientific notation as 1 +/- small deviation."""
    if pd.isna(value):
        return "NaN"
    deviation = value - 1
    if np.isclose(deviation, 0, atol=1e-50):
        return "$1.0$	"
    elif deviation > 0:
        base, exponent = f"{deviation:.1e}".split("e")
        return f"$1.0 + {float(base):.1f} \\times 10^{{{int(exponent)}}}$"
    else:
        base, exponent = f"{abs(deviation):.1e}".split("e")
        return f"$1.0 - {float(base):.1f} \\times 10^{{{int(exponent)}}}$"

def scientific_notation_around_zero(value):
    """Format a number close to 0 in scientific notation"""
    if pd.isna(value):
        return "NaN"
    if np.isclose(value, 0, atol=1e-50):
        return "$0.0$	"
    else:
        base, exponent = f"{value:.1e}".split("e")
        return f"${float(base):.1f} \\times 10^{{{int(exponent)}}}$"

def generate_latex_table(data, data_delta):
    # Convert dictionary to DataFrame, handling unequal lengths by filling with NaN
    max_length = max(len(v) for v in data.values())
    padded_data = {key: values + [np.nan] * (max_length - len(values)) for key, values in data.items()}
    df = pd.DataFrame(padded_data)

    max_length_delta = max(len(v) for v in data_delta.values())
    padded_data_delta = {key: values + [np.nan] * (max_length_delta - len(values)) for key, values in data_delta.items()}
    df_delta = pd.DataFrame(padded_data_delta)
    
    summary_stats = df.agg(['min', 'max', 'median']).transpose()
    summary_stats_delta= df_delta.agg(['max', 'mean']).transpose()
    
    # Generate the LaTeX table
    latex_table = "\\begin{table}[h!]\n\\centering\n\\begin{tabular}{|l|c|c|c|c|}\n\\hline\n"
    latex_table += "\\textbf{Dataset} & \textbf{Min($b_L$)} & \textbf{Max($b_U$)} & \textbf{Max($b_U - b_L$)} & \textbf{Mean($b_U - b_L$)} \\\\ \n\\hline\n"
    for col in summary_stats.index:
        min_val = scientific_notation_around_one(summary_stats.loc[col, 'min'])
        max_val = scientific_notation_around_one(summary_stats.loc[col, 'max'])
        max_val_delta = scientific_notation_around_zero(summary_stats_delta.loc[col, 'max'])
        mean_val_delta = scientific_notation_around_zero(summary_stats_delta.loc[col, 'mean'])
        latex_table += f"\\texttt{{{col}}} & {min_val} & {max_val} & {max_val_delta} & {mean_val_delta} \\\\ \n"
    latex_table += "\\hline\n\\end{tabular}\n\\caption{Summary Statistics for Each Dataset}\n\\label{tab:summary_stats}\n\\end{table}"

    return latex_table



In [10]:
print(generate_latex_table(dict_bounds,dict_delta_bounds))

\begin{table}[h!]
\centering
\begin{tabular}{|l|c|c|c|c|}
\hline
\textbf{Dataset} & 	extbf{Min($b_L$)} & 	extbf{Max($b_U$)} & 	extbf{Max($b_U - b_L$)} & 	extbf{Mean($b_U - b_L$)} \\ 
\hline
\texttt{MSWML} & $1.0 - 7.6 \times 10^{-4}$ & $1.0 + 1.4 \times 10^{-3}$ & $2.2 \times 10^{-3}$ & $1.9 \times 10^{-4}$ \\ 
\texttt{Polyp} & $1.0 - 1.4 \times 10^{-3}$ & $1.0 + 1.2 \times 10^{-3}$ & $2.6 \times 10^{-3}$ & $1.3 \times 10^{-5}$ \\ 
\texttt{Optic Cup} & $1.0 - 3.2 \times 10^{-4}$ & $1.0 + 5.7 \times 10^{-4}$ & $8.9 \times 10^{-4}$ & $8.5 \times 10^{-6}$ \\ 
\hline
\end{tabular}
\caption{Summary Statistics for Each Dataset}
\label{tab:summary_stats}
\end{table}


In [11]:
import numpy as np
import pandas as pd

def scientific_notation_around_one(value):
    """Format a number close to 1 in scientific notation as 1 +/- small deviation."""
    if pd.isna(value):
        return "NaN"
    deviation = value - 1
    if np.isclose(deviation, 0, atol=1e-50):
        return "$1.0$	"
    elif deviation > 0:
        base, exponent = f"{deviation:.1e}".split("e")
        return f"$1.0 + {float(base):.1f} \\times 10^{{{int(exponent)}}}$"
    else:
        base, exponent = f"{abs(deviation):.1e}".split("e")
        return f"$1.0 - {float(base):.1f} \\times 10^{{{int(exponent)}}}$"

def scientific_notation_around_zero(value):
    """Format a number close to 0 in scientific notation"""
    if pd.isna(value):
        return "NaN"
    if np.isclose(value, 0, atol=1e-50):
        return "$0.0$	"
    else:
        base, exponent = f"{value:.1e}".split("e")
        return f"${float(base):.1f} \\times 10^{{{int(exponent)}}}$"

def generate_latex_table(data):
    # Convert dictionary to DataFrame, handling unequal lengths by filling with NaN
    max_length = max(len(v) for v in data.values())
    padded_data = {key: values + [np.nan] * (max_length - len(values)) for key, values in data.items()}
    df = pd.DataFrame(padded_data)
    
    summary_stats= df.agg(['max', 'mean']).transpose()
    
    # Generate the LaTeX table
    latex_table = "\\begin{table}[h!]\n\\centering\n\\begin{tabular}{|l|c|c|}\n\\hline\n"
    latex_table += "\\textbf{Dataset} & \\textbf{Max($\epsilon$)}  & \\textbf{Mean($\epsilon$)} \\\\ \n\\hline\n"
    for col in summary_stats.index:
        max_val = scientific_notation_around_zero(summary_stats.loc[col, 'max'])
        mean_val= scientific_notation_around_zero(summary_stats.loc[col, 'mean'])
        latex_table += f"\\texttt{{{col}}} & {max_val} & {mean_val} \\\\ \n"
    latex_table += "\\hline\n\\end{tabular}\n\\caption{Summary Statistics for Each Dataset}\n\\label{tab:summary_stats}\n\\end{table}"

    return latex_table



In [12]:
print(generate_latex_table(dict_epsilon_bounds))

\begin{table}[h!]
\centering
\begin{tabular}{|l|c|c|}
\hline
\textbf{Dataset} & \textbf{Max($\epsilon$)}  & \textbf{Mean($\epsilon$)} \\ 
\hline
\texttt{MSWML} & $1.4 \times 10^{-3}$ & $1.2 \times 10^{-4}$ \\ 
\texttt{Polyp} & $1.4 \times 10^{-3}$ & $7.6 \times 10^{-6}$ \\ 
\texttt{Optic Cup} & $5.7 \times 10^{-4}$ & $5.3 \times 10^{-6}$ \\ 
\hline
\end{tabular}
\caption{Summary Statistics for Each Dataset}
\label{tab:summary_stats}
\end{table}


In [13]:
dict_epsilon_bounds

defaultdict(list,
            {'MSWML': [5.55155696035925e-06,
              0.0001242806169419186,
              0.00019770718775635743,
              3.958890745447974e-06,
              1.4549658448936853e-05,
              7.233179883492369e-06,
              9.409561698703328e-07,
              1.789507117155864e-06,
              4.320046998129712e-06,
              5.325997410654537e-06,
              3.1048805162470217e-06,
              2.898597025446925e-05,
              9.667018183412424e-05,
              1.3775426426221316e-06,
              1.8885431384729756e-05,
              2.915715276552433e-06,
              3.818146554923629e-06,
              8.742879663126146e-06,
              0.00011710212862259262,
              5.3899116574385175e-06,
              8.603839128040747e-06,
              9.013899147114479e-05,
              9.524074703337604e-06,
              0.0008994878501034087,
              0.0014291852932141191],
             'Polyp': [1.946557093290835e

In [14]:
import pandas as pd
x = np.inf
for i in dict_upper_bounds:
    y = min(dict_upper_bounds[i])
    if y<x:
        x == y
print(y)

1.000000138213191


In [15]:
import pandas as pd
x = -np.inf
for i in dict_upper_bounds:
    y = max(dict_upper_bounds[i])
    if y>x:
        x == y
print(y)

1.0005708304382595


In [16]:
import pandas as pd
x = np.inf
for i in dict_upper_bounds:
    y = min(dict_lower_bounds[i])
    if y<x:
        x == y
print(y)

0.9996782217785701


In [17]:
import pandas as pd
x = -np.inf
for i in dict_upper_bounds:
    y = max(dict_lower_bounds[i])
    if y>x:
        x == y
print(y)

0.9999996535450217
